In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Au fost detectate {len(gpus)} GPU-uri active:")
    for i, gpu in enumerate(gpus):
        print(f" Placa {i}: {gpu.name}")
else:
    print("Niciun GPU detectat. Sistemul rulează lent pe CPU.")

Au fost detectate 2 GPU-uri active:
 Placa 0: /physical_device:GPU:0
 Placa 1: /physical_device:GPU:1


In [13]:
!pip install ai_edge_litert
from ai_edge_litert.interpreter import Interpreter

In [4]:
strategy = tf.distribute.MirroredStrategy()
print(f"Numărul de unități grafice sincronizate: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Numărul de unități grafice sincronizate: 2


In [5]:
from tensorflow import keras

modele = [
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/mobilenet_gpu_fer-2013-augmented.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_ck.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_fer-2013-original.keras",
    "/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vrescnn_gpu_raf-db.keras"
]

for path in modele:
    model = keras.models.load_model(path)
    print(f"{path.split('/')[-1]} → input shape: {model.input_shape}")

mobilenet_gpu_fer-2013-augmented.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_ck.keras → input shape: (None, 48, 48, 1)
vcnn_gpu_fer-2013-original.keras → input shape: (None, 48, 48, 1)
vrescnn_gpu_raf-db.keras → input shape: (None, 48, 48, 1)


In [37]:
from tensorflow import keras
from tensorflow.keras.models import load_model
import tensorflow as tf

model = load_model('/kaggle/input/datasets/mindrocrobert/modele-emotii-faciale/vcnn_gpu_ck.keras')   
model.export('some', format="tf_saved_model")

mod=tf.saved_model.load('some')

INFO:tensorflow:Assets written to: some/assets


INFO:tensorflow:Assets written to: some/assets


Saved artifact at 'some'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 48, 48, 1), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  135476178325520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560724432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560724624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560725200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560725968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560723472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560723664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560725776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560724816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560723280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135479560721936: TensorSpec

In [38]:
converter = tf.lite.TFLiteConverter.from_keras_model(mod)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_file='vcnn_gpu_ck.tflite'
with open(tflite_file, 'wb') as f:
    f.write(tflite_model)

print("Conversia a fost realizată cu succes și modelul a fost salvat ca: ",tflite_file)

INFO:tensorflow:Assets written to: /tmp/tmplf5n01hv/assets


INFO:tensorflow:Assets written to: /tmp/tmplf5n01hv/assets
W0000 00:00:1778164876.192804      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778164876.192844      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


Conversia a fost realizată cu succes și modelul a fost salvat ca:  vcnn_gpu_ck.tflite


In [42]:
interpreter = Interpreter(model_path="vcnn_gpu_ck.tflite") 
interpreter.allocate_tensors()

In [43]:
# celula 1 - preprocesare CK+ cu split corect train/val/test
import keras
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img, img_to_array

ckplus_dir = '/kaggle/input/datasets/shawon10/ckplus/CK+48'  

print("Versiune TensorFlow: ", tf.__version__)
print("Versiune Keras: ", keras.__version__)

img_size = 48
batch_size = 16
epoci = 50
num_classes = 7

# ── Citim toate imaginile manual ──
imagini = []
etichete = []
clase = sorted(os.listdir(ckplus_dir))
print("Clase găsite:", clase)

for idx, clasa in enumerate(clase):
    folder = os.path.join(ckplus_dir, clasa)
    for fisier in os.listdir(folder):
        cale = os.path.join(folder, fisier)
        img = load_img(cale, target_size=(img_size, img_size), color_mode='grayscale')
        imagini.append(img_to_array(img) / 255.0)
        etichete.append(idx)

X = np.array(imagini)
y = np.array(etichete)

# ── Split 70% train / 15% val / 15% test ──
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

y_train = to_categorical(y_train, num_classes)
y_val   = to_categorical(y_val,   num_classes)
y_test  = to_categorical(y_test,  num_classes)

print(f"\nSetul de antrenare: {X_train.shape[0]} imagini")
print(f"Setul de validare:  {X_val.shape[0]} imagini")
print(f"Setul de testare:   {X_test.shape[0]} imagini")

# ── Augmentare doar pe train ──
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow(X_train, y_train, batch_size=batch_size, shuffle=True)
val_generator   = datagen.flow(X_val,   y_val,   batch_size=batch_size, shuffle=False)
test_generator  = datagen.flow(X_test,  y_test,  batch_size=batch_size, shuffle=False)

Versiune TensorFlow:  2.19.0
Versiune Keras:  3.10.0
Clase găsite: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']

Setul de antrenare: 686 imagini
Setul de validare:  147 imagini
Setul de testare:   148 imagini


In [45]:
import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]


y_true = np.argmax(y_test, axis=1)  
predictions = []
durata_totala = 0

for img in X_test:
    img_exp = np.expand_dims(img, axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, img_exp)
    t1 = time.time()
    interpreter.invoke()
    t2 = time.time()
    durata_totala += (t2 - t1)
    output = interpreter.get_tensor(output_index)
    predictions.append(np.argmax(output))

acc = sum(p == t for p, t in zip(predictions, y_true)) / len(y_true)
latenta = 1000 * durata_totala / len(y_true)

print(f"Acuratețe VCNN CK+ (.tflite): {acc*100:.2f}%")
print(f"Latența per imagine: {latenta:.4f} ms")
print(confusion_matrix(y_true, predictions))
print(classification_report(y_true, predictions, target_names=clase))

Acuratețe VCNN CK+ (.tflite): 90.54%
Latența per imagine: 4.5522 ms
[[15  2  1  0  0  3  0]
 [ 0  8  0  0  0  0  0]
 [ 1  1 24  0  1  0  0]
 [ 0  0  0  8  1  0  2]
 [ 0  0  0  0 31  0  0]
 [ 1  0  0  0  0 11  0]
 [ 0  1  0  0  0  0 37]]
              precision    recall  f1-score   support

       anger       0.88      0.71      0.79        21
    contempt       0.67      1.00      0.80         8
     disgust       0.96      0.89      0.92        27
        fear       1.00      0.73      0.84        11
       happy       0.94      1.00      0.97        31
     sadness       0.79      0.92      0.85        12
    surprise       0.95      0.97      0.96        38

    accuracy                           0.91       148
   macro avg       0.88      0.89      0.88       148
weighted avg       0.91      0.91      0.90       148



In [46]:
import os

size_in_bytes = os.path.getsize('vcnn_gpu_ck.tflite')
lite_sz = round(size_in_bytes/1000, 2)

print('Raport final VCNN CK+:', 
      'Lite_sz', str(lite_sz)+'k',
      'acc:', str(round(100*acc, 2)),
      'lat', str(round(latenta, 2)))

Raport final VCNN CK+: Lite_sz 1610.28k acc: 90.54 lat 4.55
